# 512

In [74]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path
import numpy as np
import pandas as pd

RESULTS_DIR = Path("evals/experiments")
PLOTS_DIR   = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

K_VALUES = [3, 4, 5, 6]
ALL_METRICS = [
    "answer_correctness",
    "faithfulness",
    "response_relevancy",
    "context_precision",
    "context_recall",
]
NO_ANSWARE_METRICS = ["faithfulness", "context_precision", "context_recall"]

INDEX_LABELS_512 = {
    "sentence":              "Sliding-window",
    "markdown":              "Markdown",
    "markdown_and_sentence": "Hybrid",
}
INDEX_LABELS_1024 = {
    "sentence":              "Sentence (sliding-window) 1024",
    "markdown":              "Markdown",
    "markdown_and_sentence": "Hybrid (Markdown + Sentence) 1024",
}
INDEX_COLORS = {
    "sentence":              "#648FFF",
    "markdown":              "#FE6100",
    "markdown_and_sentence": "#785EF0",
}
INDEX_TYPES = list(INDEX_COLORS.keys())


def build_filename(index_type: str, k: int, chunk_size: int, no_answare: bool = False) -> Path:
    na   = "_no_answare" if no_answare else ""
    base = RESULTS_DIR / ("no_answare" if no_answare else "")
    if index_type == "sentence":
        name = f"from_index_sentence{na}_{chunk_size}_k_{k}_results.csv"
    elif index_type == "markdown":
        name = f"from_index_markdown{na}_k_{k}_results.csv"
    else:
        name = f"from_index_markdown_and_sentence{na}_{chunk_size}_k_{k}_results.csv"
    return base / name


def load_all_results(chunk_size: int, no_answare: bool = False) -> pd.DataFrame:
    frames = []
    for index_type in INDEX_TYPES:
        for k in K_VALUES:
            path = build_filename(index_type, k, chunk_size, no_answare)
            if not path.exists():
                print(f"[WARNING] File not found, skipping: {path}")
                continue
            df = pd.read_csv(path).dropna(subset=["question"])
            df["index_type"] = index_type
            df["k"]          = k
            frames.append(df)
    if not frames:
        raise FileNotFoundError("No CSVs found.")
    return pd.concat(frames, ignore_index=True)


def pass_stats(df: pd.DataFrame) -> pd.DataFrame:
    grp   = df.groupby(["index_type", "k"])
    stats = grp["judge_result"].apply(
        lambda s: (s.str.lower() == "pass").sum()
    ).rename("pass_count").reset_index()
    stats["total"]     = grp["judge_result"].count().values
    stats["pass_rate"] = stats["pass_count"] / stats["total"] * 100
    return stats


def metric_means(df: pd.DataFrame, metrics: list[str], filter_absent: bool = False) -> pd.DataFrame:
    available = [m for m in metrics if m in df.columns]
    if filter_absent:
        df = df[df["answer_source"] != "absent for choice"]
    return df.groupby(["index_type", "k"])[available].mean().reset_index()


def plot_metrics_boxplots(
    df: pd.DataFrame,
    save_path: Path,
    metrics: list[str],
    index_labels: dict,
    suptitle: str,
) -> None:
    available = [m for m in metrics if m in df.columns]
    ncols = 3
    nrows = (len(available) + ncols - 1) // ncols

    fig = make_subplots(
        rows=nrows,
        cols=ncols,
        subplot_titles=[m.replace("_", " ").title() for m in available],
        vertical_spacing=0.18,
        horizontal_spacing=0.08,
    )

    rng = np.random.default_rng(42)

    for ax_idx, metric in enumerate(available):
        row = ax_idx // ncols + 1
        col = ax_idx % ncols + 1

        x_labels = [index_labels[t] for t in INDEX_TYPES]

        for index_type in INDEX_TYPES:
            vals  = df[df["index_type"] == index_type][metric].dropna().values
            color = INDEX_COLORS[index_type]
            label = index_labels[index_type]

            # --- Boxplot trace ---
            fig.add_trace(
                go.Box(
                    y=vals,
                    name=label,
                    x0=label,
                    marker_color=color,
                    fillcolor=color,
                    opacity=0.7,
                    boxmean=False,
                    showlegend=False,           # legend removed
                    legendgroup=label,
                    boxpoints=False,
                    line=dict(color=color, width=1.5),
                ),
                row=row, col=col,
            )

            # --- Jittered scatter overlay ---
            jitter = rng.uniform(-0.25, 0.25, size=len(vals))
            fig.add_trace(
                go.Scatter(
                    x=[label] * len(vals),
                    y=vals,
                    mode="markers",
                    marker=dict(color=color, size=4, opacity=0.3, line=dict(width=0)),
                    showlegend=False,
                    legendgroup=label,
                ),
                row=row, col=col,
            )

        # --- Axis formatting ---
        axis_idx  = "" if (row == 1 and col == 1) else ax_idx + 1
        xaxis_key = f"xaxis{axis_idx if axis_idx != 1 else ''}"
        yaxis_key = f"yaxis{axis_idx if axis_idx != 1 else ''}"

        fig.update_layout(**{
            xaxis_key: dict(
                tickangle=0,
                tickfont=dict(size=11),
                title=dict(text="<b>Chunking Technique</b>", font=dict(size=12)),
                categoryorder="array",
                categoryarray=x_labels,
                showline=False,
                linecolor="rgba(0,0,0,1)",
                linewidth=1,
                mirror=False,
            ),
            yaxis_key: dict(
                range=[0.0, 1.1],
                tickvals=[0, 0.25, 0.5, 0.75, 1.0],
                tickfont=dict(size=10),
                gridcolor="rgba(0,0,0,0.1)",
                gridwidth=1,
                showgrid=True,
                showline=False,
                linecolor="rgba(0,0,0,1)",
                linewidth=1,
                mirror=False,
                ticklabelstandoff=10,
            ),
        })
    # --- "Score" annotation top-left of each subplot ---
    for ax_idx in range(len(available)):
        n = "" if ax_idx == 0 else ax_idx + 1
        fig.add_annotation(
            text="<b>Score</b>",
            xref=f"x{n} domain",
            yref=f"y{n} domain",
            x=-0.06,
            y=1,
            showarrow=False,
            font=dict(size=11, color="#444"),
            xanchor="left",
            yanchor="bottom",
        )

    # --- Hide unused subplots ---
    for ax_idx in range(len(available), nrows * ncols):
        n = ax_idx + 1
        fig.update_layout(**{
            f"xaxis{n}": dict(visible=False),
            f"yaxis{n}": dict(visible=False),
        })

    fig.update_layout(
        title=dict(
            text=suptitle,
            font=dict(size=14, weight="bold"),
            x=0.5,
            xanchor="center",
        ),
        height=420 * nrows,
        width=1200,
        paper_bgcolor="white",
        plot_bgcolor="white",
        showlegend=False,                       # legend removed globally
        margin=dict(t=80, b=60, l=60, r=40),
    )

    # Remove built-in y-axis titles (replaced by annotations)
    for i in range(1, nrows * ncols + 1):
        key = f"yaxis{'' if i == 1 else i}"
        fig.update_layout(**{key: dict(title="")})

    fig.write_image(str(save_path), scale=2)
    fig.show()
    print(f"Saved: {save_path}")


def plot_pass_rate(stats: pd.DataFrame, save_path: Path, title: str, index_labels: dict) -> None:
    fig = go.Figure()

    for idx_type in INDEX_TYPES:
        subset = stats[stats["index_type"] == idx_type].sort_values("k")
        values = subset["pass_rate"].values
        x_labels = [f"k={k}" for k in subset["k"].values]

        fig.add_trace(
            go.Bar(
                x=x_labels,
                y=values,
                name=index_labels[idx_type],
                marker_color=INDEX_COLORS[idx_type],
                text=[f"{v:.0f}%" for v in values],
                textposition="outside",
                textfont=dict(size=10),
            )
        )

    fig.add_annotation(
        text="<b>Pass rate (%)</b>",
        xref="paper", yref="paper",
        x=0.04, y=1.0,
        showarrow=False,
        font=dict(size=11, color="#444"),
        xanchor="right",
        yanchor="bottom",
    )

    fig.update_layout(
        title=dict(
            text=title,
            font=dict(size=14, weight="bold"),
            x=0.5,
            xanchor="center",
        ),
        barmode="group",
        xaxis=dict(
            title=dict(text="<b>Top k chunks retrieved</b>", font=dict(size=12)),
            tickangle=0,
            tickfont=dict(size=12),
            showline=True,
            linecolor="rgba(0,0,0,0.5)",
            linewidth=1,
        ),
        yaxis=dict(
            # title=dict(text="<b>Pass rate (%)</b>", font=dict(size=12)),
            range=[0, 115],          # extra space for the text labels above bars
            tickvals=list(range(0, 101, 20)),
            tickfont=dict(size=10),
            gridcolor="rgba(0,0,0,0.08)",
            gridwidth=1,
            showgrid=True,
            showline=True,
            linecolor="rgba(0,0,0,0.5)",
            linewidth=1,
            ticklabelstandoff=6,
        ),
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
            font=dict(size=11),
        ),
        height=450,
        width=750,
        paper_bgcolor="white",
        plot_bgcolor="white",
        margin=dict(t=100, b=60, l=80, r=40),
    )

    fig.write_image(str(save_path), scale=2)
    fig.show()
    print(f"Saved: {save_path}")


# ── Main ──────────────────────────────────────────────────────────────────────
df    = load_all_results(chunk_size=512)
stats = pass_stats(df)
means = metric_means(df, ALL_METRICS, filter_absent=True)

CHUNK_SIZE = 512
df    = load_all_results(chunk_size=CHUNK_SIZE)
stats = pass_stats(df)
means = metric_means(df, ALL_METRICS, filter_absent=True)

plot_pass_rate(stats, PLOTS_DIR / "1_pass_rate.png",
               f"Judge Pass Results by Index Type and k (chunk size {CHUNK_SIZE})", INDEX_LABELS)


# plot_metrics_boxplots(
#     df,
#     PLOTS_DIR / "3_metrics_boxplots.png",
#     ALL_METRICS,
#     INDEX_LABELS_512,
#     "RAGAS Metrics Distribution by Index Type — chunk size 512",
# )

Saved: plots/1_pass_rate.png


# PER OGNI K

In [71]:
import plotly.graph_objects as go
from pathlib import Path
import numpy as np
import pandas as pd

RESULTS_DIR = Path("evals/experiments")
PLOTS_DIR   = Path("plots")
PLOTS_DIR.mkdir(exist_ok=True)

K_VALUES = [3, 4, 5, 6]
ALL_METRICS = [
    "answer_correctness",
    "faithfulness",
    "response_relevancy",
    "context_precision",
    "context_recall",
]

INDEX_LABELS = {
    "sentence":              "Sliding-window",
    "markdown":              "Structured",
    "markdown_and_sentence": "Hybrid",
}
INDEX_COLORS = {
    "sentence":              "#648FFF",
    "markdown":              "#FE6100",
    "markdown_and_sentence": "#785EF0",
}
INDEX_TYPES = list(INDEX_COLORS.keys())


def build_filename(index_type: str, k: int, chunk_size: int, no_answare: bool = False) -> Path:
    na   = "_no_answare" if no_answare else ""
    base = RESULTS_DIR / ("no_answare" if no_answare else "")
    if index_type == "sentence":
        name = f"from_index_sentence{na}_{chunk_size}_k_{k}_results.csv"
    elif index_type == "markdown":
        name = f"from_index_markdown{na}_k_{k}_results.csv"
    else:
        name = f"from_index_markdown_and_sentence{na}_{chunk_size}_k_{k}_results.csv"
    return base / name


def load_all_results(chunk_size: int, no_answare: bool = False) -> pd.DataFrame:
    frames = []
    for index_type in INDEX_TYPES:
        for k in K_VALUES:
            path = build_filename(index_type, k, chunk_size, no_answare)
            if not path.exists():
                print(f"[WARNING] File not found, skipping: {path}")
                continue
            df = pd.read_csv(path).dropna(subset=["question"])
            df["index_type"] = index_type
            df["k"]          = k
            frames.append(df)
    if not frames:
        raise FileNotFoundError("No CSVs found.")
    return pd.concat(frames, ignore_index=True)


def plot_single_metric_boxplot(
    df: pd.DataFrame,
    save_path: Path,
    metric: str,
    index_labels: dict,
    suptitle: str,
) -> None:
    if metric not in df.columns:
        print(f"[WARNING] Metric '{metric}' not found, skipping.")
        return

    fig = go.Figure()
    rng = np.random.default_rng(42)

    for index_type in INDEX_TYPES:
        label = index_labels[index_type]
        color = INDEX_COLORS[index_type]

        x_all, y_all = [], []
        for k in K_VALUES:
            vals = df[
                (df["index_type"] == index_type) & (df["k"] == k)
            ][metric].dropna().values
            x_all.extend([f"k={k}"] * len(vals))
            y_all.extend(vals.tolist())

        fig.add_trace(
            go.Box(
                x=x_all,
                y=y_all,
                name=label,
                marker_color=color,
                fillcolor=color,
                opacity=0.7,
                boxmean=False,
                showlegend=True,
                legendgroup=label,
                boxpoints="all",
                jitter=0.4,
                pointpos=0,
                marker=dict(color=color, size=4, opacity=0.25, line=dict(width=0)),
                line=dict(color=color, width=1.5),
            )
        )

    fig.add_annotation(
        text="<b>Score</b>",
        xref="paper", yref="paper",
        x=-0.02, y=1.0,
        showarrow=False,
        font=dict(size=11, color="#444"),
        xanchor="right",
        yanchor="bottom",
    )

    fig.add_annotation(
        text="<b>Pass rate (%)</b>",
        xref="paper", yref="paper",
        x=-0.02, y=1.0,
        showarrow=False,
        font=dict(size=11, color="#444"),
        xanchor="right",
        yanchor="bottom",
    )

    fig.update_layout(
        title=dict(
            text=suptitle,
            font=dict(size=14, weight="bold"),
            x=0.5,
            xanchor="center",
        ),
        xaxis=dict(
            tickangle=0,
            tickfont=dict(size=12),
            title=dict(text="<b>Top k chunks retrieved</b>", font=dict(size=12)),
            categoryorder="array",
            categoryarray=[f"k={k}" for k in K_VALUES],
            showline=False,
            linecolor="rgba(0,0,0,0.5)",
            linewidth=1,
        ),
        yaxis=dict(
            range=[0.0, 1.1],
            tickvals=[0, 0.25, 0.5, 0.75, 1.0],
            tickfont=dict(size=10),
            gridcolor="rgba(0,0,0,0.08)",
            gridwidth=1,
            showgrid=True,
            showline=False,
            linecolor="rgba(0,0,0,0.5)",
            linewidth=1,
            ticklabelstandoff=6,
            # title="",
        ),
        boxmode="group",
        boxgap=0.2,
        boxgroupgap=0.1,
        height=450,
        width=900,
        paper_bgcolor="white",
        plot_bgcolor="white",
        legend=dict(
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1,
            font=dict(size=11),
        ),
        margin=dict(t=100, b=60, l=80, r=40),
    )

    fig.write_image(str(save_path), scale=2)
    fig.show()
    print(f"Saved: {save_path}")


# ── Main ──────────────────────────────────────────────────────────────────────
df = load_all_results(chunk_size=512)

plot_pass_rate(stats, PLOTS_DIR / "1_pass_rate.png",
               f"Judge Pass Results by Index Type and k (chunk size {CHUNK_SIZE})", INDEX_LABELS)

# for metric in ALL_METRICS:
#     plot_single_metric_boxplot(
#         df,
#         PLOTS_DIR / f"boxplot_{metric}.png",
#         metric,
#         INDEX_LABELS,
#         f"{metric.replace('_', ' ').title()} — chunk size 512",
#     )

Saved: plots/1_pass_rate.png
